## Testing pipelines

### Research Pipeline

I start with all the steps packaged so far leading to logging a new role of trials
- running the backtest engine
- take the NAV and turnover
- run the performance calulator
- build the new row with all the information
- return the row

These steps will be repeated with each different allocator we will build from now on so I decide to package them into a pipeline now  

In [27]:
# Imports

import pandas as pd
import numpy as np
import tempfile
from pathlib import Path
from vportfolio import allocators, engine,  metrics, trials
from vportfolio.params import CAP, LOOKBACK, BACKTEST, TRIALS_LOG

In [8]:
# chain the functions into a pipeline

# each function will feed into each other to deliver the final new row for the log
def trial_pipeline(name, allocator, role, data, cap=CAP, lookback=LOOKBACK, folder=BACKTEST, trials_log = TRIALS_LOG):
    # run the engine
    result = engine.backtest_engine(allocator, 
                                    data["close"], data["open"], data["returns"], 
                                    data["eligible"], data["rebal_dates"],
                                    data["cost_bp"],
                                    cap, lookback)
    # save NAV and turnover
    nav = result["NAV"]
    turnover = result["Turnover"]
    # run the perofrmance calculator
    performance = metrics.performance_calculator(nav, data["rf_daily"])
    # save the NAV returns in parquet
    path = trials.save_returns(name, nav, folder)
    # make the new row to be logged with all parameters
    new_row = trials.make_new_row(name, allocator, role, performance, turnover, path, cap,lookback)
    # update the logs
    trials.update_trials_log(pd.DataFrame([new_row]), trials_log)
    return new_row # return the information about the trial run

In [32]:
# function testing

# create temp folder
tmp = Path(tempfile.mkdtemp())

# set variables for a sixty_forty run
dates = pd.bdate_range("2020-01-01", periods=10)
name = "sixty_forty"
role = "benchmark"
close = pd.DataFrame({"SPY": 100 * 1.01 ** np.arange(10),
                      "AGG": 100.0}, index=dates)
open_ = close.copy()
data = {"close": close,
        "open": open_,
        "returns": close.pct_change(fill_method=None),
        "eligible": close.notna(),
        "rebal_dates": pd.Series([dates[2], dates[5]]),
        "cost_bp": pd.Series(2.0, index=close.columns),
        "rf_daily": pd.Series(0.0, index=dates)}

In [48]:
# make a test run with 6040 allocator

new_row = trial_pipeline("test_6040", allocators.sixty_forty, "benchmark", data, cap = None, lookback = 1, folder = tmp, trials_log = tmp/"log.csv")
log = pd.read_csv(tmp / "log.csv")
assert len(log) == 1 # check that the log has 1 new row
saved = log.iloc[0]
# iterate over every line in both new_row and the saved log to make sure they are the same 
for key in new_row:
    if isinstance(new_row[key], str):
        assert saved[key] == new_row[key], key # if is text, check line by line all key match. add key at the end in case there is an error
    else:
        assert np.isclose(saved[key], new_row[key]), key # otherwise it's a number so check if they are close
        
print("Pipeline saved everything correctly")


Pipeline saved everything correctly
